In [ ]:
from dataclasses import dataclass
from typing import Optional
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ukf import *
import torch
from torch import nn
from functools import partial

torch.set_default_dtype(torch.float32)
torch.set_default_device('cpu')

from torchdiffeq import odeint
torch.manual_seed(42)

ODE_STEP     = 1/4    # RK4 fixed step for the u-ODE

In [ ]:
discounts   = pd.read_excel('discounts.xlsx', index_col='date').dropna()
discounts = discounts[4200:]
panel_data  = pd.read_excel('macrodata.xlsx',  index_col='meeting_date').dropna()

tenors_np          = np.asarray(discounts.columns[1:])  # M tenors
obs_discounts_np   = discounts.to_numpy()
discount_curves_np = obs_discounts_np[:, 1:]
short_rate_np      = ((1.0 / obs_discounts_np[:, 0]) - 1.0) * 360.0
times_np           = np.asarray((discounts.index - discounts.index[0]).days / 360.0)

meeting_dates      = pd.to_datetime(panel_data.index)
meeting_times_np   = np.asarray((meeting_dates - discounts.index[0]).days / 360.0)

In [ ]:
# to torch
tenors         = torch.tensor(tenors_np)
discount_curves= torch.tensor(discount_curves_np)
short_rate     = torch.tensor(short_rate_np)
meeting_times  = torch.tensor(meeting_times_np)
times          = torch.tensor(times_np)

short_rate     = short_rate.float()
discount_curves = discount_curves.float()
tenors         = tenors.float()
times          = times.float()
meeting_times  = meeting_times.float()

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_input, n_hidden, n_out, n_layers=2, activation=nn.Tanh):
        super().__init__()
        layers = [nn.Linear(n_input, n_hidden), activation()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(n_hidden, n_hidden), activation()]
        layers += [nn.Linear(n_hidden, n_out)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
class ODEModel(nn.Module):
    def __init__(self, net: nn.Module):
        super().__init__()
        self.net = net
        self.t: Optional[torch.Tensor] = None
        self.x: Optional[torch.Tensor] = None

    def set_state(self, t: torch.Tensor, x: torch.Tensor):
        self.t = t
        self.x = x

    def forward(self, tau, u):
        if tau.dim() == 0:
            tau_ = tau.reshape(-1, 1)
        else:
            tau_ = tau.view(-1, 1)
        B = tau_.shape[0]
        t_ = self.t.expand(B, 1)
        x_ = self.x.expand(B, 1)
        u_ = u.view(B, 1)
        inp = torch.cat([tau_, u_, t_, x_], dim=1)  # (B,4)
        return self.net(inp)

In [ ]:
class NeuralDiscounts(nn.Module):
    def __init__(self, u_ode: ODEModel, possible_jumps: torch.Tensor, jump_size: float, 
                 absolute_meeting_times: torch.Tensor, obs_meetings: int = 12, meeting_freq: torch.Tensor = torch.tensor([1 / 12])):
        super().__init__()
        self.u_ode = u_ode
        self.possible_jumps = possible_jumps
        self.jump_size = jump_size
        self.absolute_meeting_times = absolute_meeting_times
        self.obs_meetings = obs_meetings
        self.meeting_freq = meeting_freq

    def generate_meeting_times(self, t: torch.Tensor, T: torch.Tensor):
        t = float(t.squeeze())
        T = float(T.squeeze())
        freq = float(self.meeting_freq.item())

        # Real meetings after t within horizon
        rel = (self.absolute_meeting_times - t)          # τ for each calendar meeting
        mask = (rel > 0.0) & (rel <= T)
        real_tau = rel[mask]
        real_tau, _ = torch.sort(real_tau)
        if real_tau.numel() > self.obs_meetings:
            real_tau = real_tau[: self.obs_meetings]

        # If no extension needed, return real τ
        if real_tau.numel() == 0:
            # start synthetic meetings from the first interval after t
            start = min(freq, T + 1e-12)   # handle T0 < freq
            if start >= T - 1e-12:
                return torch.empty(0)
            ext = torch.arange(start, T + 1e-12, freq)
            return ext

        last_real = float(real_tau[-1].item())
        start = last_real + freq
        if start > T - 1e-12:
            return real_tau

        ext = torch.arange(start, T + 1e-12, freq)

        all_tau = torch.cat([real_tau.to(ext.dtype), ext], dim=0)
        all_tau = torch.unique(all_tau)
        all_tau, _ = torch.sort(all_tau)
        return all_tau

    def jump_probs(self, u: torch.Tensor) -> torch.Tensor:
        if u.dim() == 2:
            logits = (u / self.jump_size) * self.possible_jumps.view(1, -1)
            logits = logits - logits.amax(dim=-1, keepdim=True)
            return torch.softmax(logits, dim=-1)
        elif u.dim() == 3:
            logits = (u / self.jump_size) * self.possible_jumps.view(1, 1, -1)
            logits = logits - logits.amax(dim=-1, keepdim=True)
            return torch.softmax(logits, dim=-1)
        else:
            raise ValueError("u must be (K,1) or (K,B,1)")

    def solve_u(self, t: torch.Tensor, tau_grid: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        self.u_ode.set_state(t=t, x=x)
        u0 = torch.zeros_like(x)
        # Use rk4 with larger step size for speed
        u_path = odeint(self.u_ode, u0, tau_grid, method='rk4', options={'step_size': ODE_STEP})
        return u_path.squeeze(1)

    def solve_u_batch(self, t: torch.Tensor, tau_grid: torch.Tensor, x_batch: torch.Tensor) -> torch.Tensor:
        u0 = torch.zeros(1, 1)
        if tau_grid.dim() == 1:
            B = t.shape[0]
            K = tau_grid.shape[0]
            u_path = torch.empty(B, K)
            for i in range(B):
                self.u_ode.set_state(t=t[i].unsqueeze(0), x=x_batch[i].unsqueeze(0))
                u = odeint(self.u_ode, u0, tau_grid, method='rk4').squeeze()
                u_path[i, :] = u
            return u_path
        elif tau_grid.dim() == 2:
            B, K = tau_grid.shape
            u_path = torch.empty(B, K)
            for i in range(B):
                self.u_ode.set_state(t=t[i].unsqueeze(0), x=x_batch[i].unsqueeze(0))
                u = odeint(self.u_ode, u0, tau_grid[i, :], method='rk4').squeeze()
                u_path[i, :] = u
            return u_path
        else:
            raise ValueError("tau_grid must be (K,) or (B,K)")

    def discount(self, t: torch.Tensor, T: torch.Tensor, x: torch.Tensor, r0: torch.Tensor):
        tau_grid = self.generate_meeting_times(t, T)
        base = torch.exp(-r0 * T)
        if tau_grid.numel() == 0:
            return base

        u = self.solve_u(t, tau_grid, x)
        if torch.isnan(u).any() or torch.isinf(u).any():
            print(f"NaN/Inf in u! x={x}, t={t}, T={T}")
            return torch.tensor(0.01)  # Return something reasonable
        
        if u.dim() == 1:
            u = u.unsqueeze(-1)
        probs = self.jump_probs(u)
        
        # CHECK THIS:
        if torch.isnan(probs).any():
            print(f"NaN in probs! u={u}")
            return torch.tensor(0.01)
        
        djs = torch.exp(-self.possible_jumps.view(1, -1) * (T - tau_grid).view(-1, 1))
        jump_discounts = (djs * probs).sum(dim=1)
        result = base * jump_discounts.prod()
        return result
    
    def discount_curve(self, t: torch.Tensor, T_vec: torch.Tensor, x: torch.Tensor, r0: torch.Tensor) -> torch.Tensor:
        if T_vec.dim() != 1:
            raise ValueError("T_vec should be 1-D (M,)")
        
        M = len(T_vec)
        base = torch.exp(-r0 * (T_vec))
        
        # Generate meeting times for all maturities at once
        all_tau_grids = []
        max_meetings = 0
        for T in T_vec:
            tau_grid = self.generate_meeting_times(t, T)
            all_tau_grids.append(tau_grid)
            max_meetings = max(max_meetings, len(tau_grid))
        
        if max_meetings == 0:
            return base
        
        # Solve u for all unique tau values at once
        unique_taus = torch.unique(torch.cat(all_tau_grids))
        if unique_taus.numel() == 0:
            return base
        
        # Single ODE solve for all tau points
        self.u_ode.set_state(t=t, x=x)
        u0 = torch.zeros_like(x)
        u_all = odeint(self.u_ode, u0, unique_taus, method='rk4').squeeze(1)
        
        # Now compute discounts for each maturity
        discounts = []
        for i, T in enumerate(T_vec):
            tau_grid = all_tau_grids[i]
            if tau_grid.numel() == 0:
                discounts.append(base[i])
                continue
            
            # Find u values for this maturity's meeting times
            # (use searchsorted for efficiency)
            indices = torch.searchsorted(unique_taus, tau_grid)
            u = u_all[indices].unsqueeze(-1)
            
            probs = self.jump_probs(u)
            djs = torch.exp(-self.possible_jumps.view(1, -1) * (T - tau_grid).view(-1, 1))
            jump_discounts = (djs * probs).sum(dim=1)
            discounts.append(base[i] * jump_discounts.prod())
        
        return torch.stack(discounts, dim=0)

    def forward_discounts(self, t: torch.Tensor, x: torch.Tensor, tau_grid: torch.Tensor) -> torch.Tensor:
        # deltas[0] = tau_grid[0] (from t to first meeting),
        # deltas[k] = tau_grid[k] - tau_grid[k-1]
        deltas = torch.empty_like(tau_grid)
        deltas[0] = tau_grid[0]
        deltas[1:] = tau_grid[1:] - tau_grid[:-1]

        # u(τ) -> probs(τ) at each meeting
        u = self.solve_u(t, tau_grid, x)              # (K,)
        if u.dim() == 1:
            u = u.unsqueeze(-1)                       # (K,1)
        probs = self.jump_probs(u)                    # (K, J)

        # per-meeting expected forward discount factor:
        # E[exp(-J * Δτ)] with J on your jump grid
        djs = torch.exp(-self.possible_jumps.view(1, -1) * deltas.view(-1, 1))   # (K, J)
        fwd_expected = (djs * probs).sum(dim=1)                                  # (K,)
        return fwd_expected

def interp1d_clamped(xq: torch.Tensor, xp: torch.Tensor, fp: torch.Tensor) -> torch.Tensor:
    """
    Linear interpolate fp(xp) onto xq. Clamps to endpoints outside [xp[0], xp[-1]].
    xp must be 1D increasing. Works on CPU/GPU.
    """
    assert xp.dim() == 1 and fp.dim() == 1
    device = xp.device
    xq = xq.to(device)
    # Left/right masks
    left  = xq <= xp[0]
    right = xq >= xp[-1]
    mid   = (~left) & (~right)

    out = torch.empty_like(xq)

    # End clamps
    out[left]  = fp[0]
    out[right] = fp[-1]

    if mid.any():
        xm = xq[mid]
        # indices so that xp[idx-1] <= xm < xp[idx]
        idx = torch.searchsorted(xp, xm, right=False)
        idx = idx.clamp(1, xp.numel()-1)

        x0 = xp[idx-1]; x1 = xp[idx]
        y0 = fp[idx-1]; y1 = fp[idx]
        w  = (xm - x0) / (x1 - x0 + 1e-12)
        out[mid] = y0 + w * (y1 - y0)

    return out
  

def extract_single_segment_observation(
    t: torch.Tensor,
    tau_next: torch.Tensor, 
    tenors: torch.Tensor,
    discount_curve: torch.Tensor,
    r0: torch.Tensor
) -> torch.Tensor:

    D_tau = interp1d_clamped(tau_next.unsqueeze(0), tenors, discount_curve).squeeze()
    D_tau_deflated = D_tau / torch.exp(-r0 * tau_next)
    return D_tau_deflated


In [ ]:
class RiskNeutralSDE(nn.Module):
    noise_type: str = 'scalar'
    sde_type: str = 'ito'

    def __init__(self, diff_net: nn.Module):
        super().__init__()
        self.diff_net = diff_net

    def f(self, t: torch.Tensor, x: torch.Tensor):
        return torch.zeros_like(x)

    def g(self, t: torch.Tensor, x: torch.Tensor):
        # diff_net(x) can be shape (B,1) or (1,) depending on x
        sigma = self.diff_net(x)
        sigma = torch.nn.functional.softplus(sigma)

        # Force (B, 1, 1) so downstream indexing is consistent
        if sigma.dim() == 0:
            sigma = sigma.view(1, 1, 1)
        elif sigma.dim() == 1:        # (B,)
            sigma = sigma.view(-1, 1, 1)
        elif sigma.dim() == 2:        # (B,1)
            sigma = sigma.view(-1, 1, 1)
        # else: assume already (B,1,1)
        return sigma


class PhysicalSDE(nn.Module):
    noise_type: str = 'scalar'
    sde_type: str = 'ito'

    def __init__(self, risk_neutral_sde: RiskNeutralSDE, drift_net: nn.Module):
        super().__init__()
        self.rn = risk_neutral_sde
        self.drift = drift_net

    def f(self, t: torch.Tensor, x: torch.Tensor):
        mu = self.drift(x).view_as(x)
        sigma = self.rn.g(t, x)[:, 0, 0:1]
        return -mu * sigma

    def g(self, t: torch.Tensor, x: torch.Tensor):
        return self.rn.g(t, x)

In [181]:
def init_mlp_weights(model: nn.Module, output_scale: float = 0.01):
    """
    Initialize MLP weights with stable small values.
    Uses orthogonal initialization for hidden layers for better gradient flow.
    """
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if m.out_features == 1:  # output layer
                # Very small initialization for output layer
                nn.init.uniform_(m.weight, -output_scale, output_scale)
                nn.init.zeros_(m.bias)
            else:
                # Orthogonal initialization for better gradient flow
                nn.init.orthogonal_(m.weight, gain=0.5)
                nn.init.zeros_(m.bias)

In [182]:
def init_mlp_weights(model: nn.Module, output_scale: float = 0.01):
    """
    Initialize MLP weights with stable small values.
    Uses orthogonal initialization for hidden layers for better gradient flow.
    """
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if m.out_features == 1:  # output layer
                # Very small initialization for output layer
                nn.init.uniform_(m.weight, -output_scale, output_scale)
                nn.init.zeros_(m.bias)
            else:
                # Orthogonal initialization for better gradient flow
                nn.init.orthogonal_(m.weight, gain=0.5)
                nn.init.zeros_(m.bias)


def train(
    short_rates: torch.Tensor,
    tenors: torch.Tensor,
    discount_data: torch.Tensor,
    x_P: PhysicalSDE,
    discounts_model: NeuralDiscounts,
    epochs: int = 50,
    dt: float = 1/360,
    times: Optional[torch.Tensor] = None,
):
    params = list(x_P.parameters()) + list(discounts_model.parameters())
    optimizer = torch.optim.Adam(params, lr=1e-3)

    N, M = discount_data.shape
    if times is None:
        times = torch.zeros(N)

    # deterministic drift (Euler step) with safety
    def fx(x):
        drift = x_P.f(torch.zeros_like(x), x) * dt
        x_t = x + drift
        return x_t

    for epoch in range(epochs):
        optimizer.zero_grad()
        x0 = torch.tensor([0.0])
        P0 = torch.eye(1) * 0.01  # Reduced uncertainty
        R  = torch.eye(M) * 1e-3

        ukf = SquareRootUnscentedKalmanFilter(
            fx=fx,
            hx=lambda _x: _x,  # will be overridden per-step
            x=x0,
            P=P0,
            Q=torch.eye(1) * 1e-4,
            R=R,
            alpha=0.1, beta=2.0, kappa=0.0,
        )

        total_ll = 0.0
        curve_fit_loss = 0.0
        for i in range(N):
            r0 = short_rates[i]
            t_abs = times[i]

            # measurement model with safety
            def hx_row(x, *, t=t_abs, r0_row=r0):
                d = discounts_model.discount_curve(t, tenors, x, r0_row)
                return d

            ukf.hx = lambda x, _hx=hx_row: _hx(x)

            # Update Q with bounds
            sigma = x_P.g(t_abs.view(1, 1), ukf.x.view(1, 1)).squeeze()
            q_scalar = (sigma ** 2) * dt
            q_scalar = torch.clamp(q_scalar, min=1e-6, max=1e-2)
            ukf.Q = q_scalar.view(1, 1)

            # observation
            z = discount_data[i, :]
            
            try:
                x, P, ll, z_pred = ukf.step(z)                
                curve_fit_loss = curve_fit_loss + torch.mean((z - z_pred) ** 2)
                
                # Safety check after step
                if torch.isnan(x).any() or torch.isinf(x).any():
                    print(f"NaN detected at step {i}, resetting filter")
                    ukf.x = torch.zeros_like(x0)
                    ukf.S = torch.linalg.cholesky(P0 + 1e-6 * torch.eye(1))
                    continue
                
                # Safety check on likelihood
                if torch.isnan(ll) or torch.isinf(ll):
                    print(f"NaN likelihood at step {i}, skipping")
                    continue
                    
                total_ll = total_ll + ll
                
                # print(f"Step {i}: x={x.item():.6f}, sigma={sigma.item():.6f}, ll={ll.item():.3f}")
                    
            except Exception as e:
                print(f"Error at step {i}: {e}")
                # Reset filter on error
                ukf.x = torch.zeros_like(x0)
                ukf.S = torch.linalg.cholesky(P0)
                continue

        # loss = -total_ll + curve_fit_loss * 1_000_000
        loss = curve_fit_loss
        
        # Check if loss is valid before backprop
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"Epoch {epoch+1}/{epochs} | Invalid loss, skipping backward pass")
            continue
            
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}/{epochs} | neg-log-likelihood = {-total_ll.item():.3f} | curve-fit-loss = {curve_fit_loss.item():.6f} | total loss = {loss.item():.3f} | last x = {ukf.x.item():.6f}")

In [183]:
def train_independent_segments(
    short_rates: torch.Tensor,
    tenors: torch.Tensor,
    discount_data: torch.Tensor,
    x_P: PhysicalSDE,
    discounts_model: NeuralDiscounts,
    epochs: int = 50,
    dt: float = 1/360,
    times: Optional[torch.Tensor] = None,
):
   
    params = list(x_P.parameters()) + list(discounts_model.parameters())
    optimizer = torch.optim.Adam(params, lr=1e-3)
    
    def fx(x):
        drift = x_P.f(torch.zeros_like(x), x) * dt
        return x + drift
    
    N = discount_data.shape[0]
    for epoch in range(epochs):
        optimizer.zero_grad()
        x0 = torch.tensor([0.0])
        P0 = torch.eye(1) * 0.01

        ukf = SquareRootUnscentedKalmanFilter(
            fx=fx, hx=None, x=x0, P=P0,
            Q=torch.eye(1) * 1e-4,
            R=torch.eye(1) * 1e-3,
            alpha=0.1, beta=2.0, kappa=0.0,
        )

        total_ll = torch.tensor(0.0)   # tensor from start
        curve_fit_loss = torch.tensor(0.0)
        for i in range(N):
            r0        = short_rates[i]
            obs_curve = discount_data[i, :]

            # make T on same device/dtype
            tau_grid = discounts_model.generate_meeting_times(
                t=times[i], T=torch.tensor([10.0])
            )
            K = int(tau_grid.numel())
            if K == 0:
                continue
            
            deltas = torch.empty_like(tau_grid)
            deltas[0]  = tau_grid[0]
            deltas[1:] = tau_grid[1:] - tau_grid[:-1]

            ukf.R = torch.eye(K) * 1e-3

            def hx_segment(x, t_abs=times[i], tg=tau_grid):
                d = discounts_model.forward_discounts(t_abs, x, tg).view(-1)
                fwds = -torch.log(d) / deltas
                return fwds

            ukf.hx = hx_segment

            # process noise
            sigma   = x_P.g(times[i].view(1,1), ukf.x.view(1,1)).squeeze()
            q_scalar = (sigma**2 * dt)
            ukf.Q    = q_scalar.view(1,1)

            D_tau_k = interp1d_clamped(tau_grid, tenors, obs_curve).view(-1)  
            D_tau_prev = torch.cat([tau_grid.new_ones(1), D_tau_k[:-1]], dim=0)  
            fwd = D_tau_k / D_tau_prev  # (K,)

            
            fwd_clean = fwd / torch.exp(-r0 * deltas)
            fwd_clean = -torch.log(fwd_clean) / deltas
            x_next, P_next, ll_step, _ = ukf.step(fwd_clean)

            z_pred = ukf.hx(ukf.x).view(-1)             
            mse    = ((fwd_clean - z_pred)**2).mean()
            curve_fit_loss = curve_fit_loss + mse
            total_ll   = total_ll - ll_step

            ukf.x = x_next
            if not (torch.isnan(x_next).any() or torch.isinf(x_next).any()):
                ukf.S = torch.linalg.cholesky(P_next)
            else:
                ukf.x = x0.clone()
                ukf.S = torch.linalg.cholesky(P0)

        loss =  curve_fit_loss * 1_000_000
        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch+1}/{epochs} | loss={float(loss.item()):.6f} | x={ukf.x.item()} | curve_fit={curve_fit_loss.item():.6f} | total_ll={total_ll.item():.3f}" )


In [184]:
ode_net = MLP(n_input=4, n_hidden=5, n_out=1, n_layers=2, activation=nn.Tanh)
u_ode = ODEModel(ode_net)

# jump grid & meeting calendar
possible_jumps = torch.tensor([-50, -25, 0.0, 25, 50])/10_000  # example grid
jump_size = 25/10_000
absolute_meeting_times = meeting_times  # from your data load

nd = NeuralDiscounts(
    u_ode=u_ode,
    possible_jumps=possible_jumps,
    jump_size=jump_size,
    absolute_meeting_times=absolute_meeting_times,
    obs_meetings=12,
    meeting_freq=torch.tensor([1/4])
)

# Risk-neutral diff & physical drift
# After defining your models
diff_net  = MLP(n_input=1, n_hidden=1, n_out=1, n_layers=1, activation=nn.Tanh)
drift_net = MLP(n_input=1, n_hidden=1, n_out=1, n_layers=1, activation=nn.Tanh)

rn_sde    = RiskNeutralSDE(diff_net)
phys_sde  = PhysicalSDE(risk_neutral_sde=rn_sde, drift_net=drift_net)

# Initialize all nets
init_mlp_weights(diff_net)
init_mlp_weights(drift_net)
init_mlp_weights(ode_net)

ten = tenors.to(torch.float32)
disc_mat = discount_curves.to(torch.float32)
sr = short_rate.to(torch.float32)
ts = times.to(torch.float32)

# Optional: downsample for a quick smoke test
max_rows = min(1, disc_mat.shape[0])
disc_mat = disc_mat[:max_rows]
sr = sr[:max_rows]
ts = ts[:max_rows]

# train(
#     short_rates=sr,
#     tenors=ten,
#     discount_data=disc_mat,
#     x_P=phys_sde,
#     discounts_model=nd,
#     epochs=2_500,          # start small
#     dt=1/360,
#     times=ts
# )

In [185]:
train_independent_segments(
    short_rates=sr,
    tenors=ten,
    discount_data=disc_mat,
    x_P=phys_sde,
    discounts_model=nd,
    epochs=2_500,          # start small
    dt=1/360,
    times=ts
)

Epoch 1/2500 | loss=2050.416748 | x=-1.0841004041139968e-05 | curve_fit=0.002050 | total_ll=-64.944
Epoch 2/2500 | loss=2050.183594 | x=7.211935553641524e-06 | curve_fit=0.002050 | total_ll=-64.945
Epoch 3/2500 | loss=2049.957520 | x=7.760076186968945e-06 | curve_fit=0.002050 | total_ll=-64.946
Epoch 4/2500 | loss=2049.748291 | x=3.108945384155959e-05 | curve_fit=0.002050 | total_ll=-64.955
Epoch 5/2500 | loss=2049.540527 | x=-5.922343916608952e-06 | curve_fit=0.002050 | total_ll=-64.955
Epoch 6/2500 | loss=2049.342773 | x=-4.0058330341707915e-06 | curve_fit=0.002049 | total_ll=-64.966
Epoch 7/2500 | loss=2049.148926 | x=2.6303496269974858e-05 | curve_fit=0.002049 | total_ll=-64.961
Epoch 8/2500 | loss=2048.966553 | x=3.085931530222297e-05 | curve_fit=0.002049 | total_ll=-64.973
Epoch 9/2500 | loss=2048.788818 | x=-4.051534051541239e-05 | curve_fit=0.002049 | total_ll=-64.971
Epoch 10/2500 | loss=2048.620605 | x=-1.2872832485300023e-05 | curve_fit=0.002049 | total_ll=-64.988
Epoch 11/2

KeyboardInterrupt: 

In [186]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def diagnose_model(
    discounts_model: NeuralDiscounts,
    x_P: PhysicalSDE,
    tenors: torch.Tensor,
    short_rates: torch.Tensor,
    times: torch.Tensor,
    discount_data: torch.Tensor,
    tau_dense_points: int = 300
):
    """
    Diagnose why the model isn't working and PLOT the learned u(τ) shape.
    """
    print("="*60)
    print("MODEL DIAGNOSTICS")
    print("="*60)
    
    # Test with a simple state
    x_test = torch.tensor([0.0])
    t_test = times[0]
    r0_test = short_rates[0]
    T_test = tenors[5]  # Pick a medium maturity
    
    print(f"\nTest inputs:")
    print(f"  x = {x_test.item():.6f}")
    print(f"  t = {t_test.item():.6f}")
    print(f"  r0 = {r0_test.item():.6f}")
    print(f"  T = {T_test.item():.6f}")
    
    with torch.no_grad():
        # Set ODE state (needed by ODEModel.forward)
        discounts_model.u_ode.set_state(t=t_test, x=x_test)
        u0 = torch.zeros_like(x_test)
        
        # --- Dense u(τ) trajectory from τ=0..T_test for plotting ---
        T_scalar = float(T_test.item())
        # make sure the first point is >0 to avoid any edge numerical issues
        tau_dense = torch.linspace(0.0, T_scalar, tau_dense_points).to(x_test.dtype)
        # integrate u-ODE over dense grid
        u_dense = odeint(discounts_model.u_ode, u0, tau_dense, method='rk4',
                         options={'step_size': ODE_STEP}).squeeze()
        
        # --- u(τ) only at MEETING times the model uses (τ-grid) ---
        tau_grid = discounts_model.generate_meeting_times(t_test, T_test)
        if tau_grid.numel() > 0:
            # Solve u at the meeting τs using the model's helper
            u_meet = discounts_model.solve_u(t_test, tau_grid, x_test)  # (K,)
            # Jump probabilities at meeting times
            u_for_probs = u_meet.unsqueeze(-1)  # (K,1)
            probs_meet = discounts_model.jump_probs(u_for_probs).detach().cpu().numpy()  # (K, |J|)
            tau_meet_np = tau_grid.detach().cpu().numpy()
        else:
            u_meet = torch.tensor([])
            probs_meet = None
            tau_meet_np = np.array([])
        
        # Quick text diagnostics for ODE sanity at a few τ values
        tau_test = torch.tensor([0.1, 0.5, 1.0]).clamp_(max=T_test.item())
        try:
            u_path = odeint(discounts_model.u_ode, u0, tau_test, method='rk4', 
                            options={'step_size': ODE_STEP})
            print(f"\n  ODE u values at tau={tau_test.numpy()}: {u_path.squeeze().numpy()}")
        except Exception as e:
            print(f"\n  ODE failed: {e}")
        
        # Check volatility & drift
        sigma = x_P.g(t_test.view(1,1), x_test.view(1,1))
        print(f"\n  Volatility σ(x={x_test.item():.3f}) = {sigma.item():.6f}")
        mu = x_P.f(t_test.view(1,1), x_test.view(1,1))
        print(f"  Drift f(x={x_test.item():.3f}) = {mu.item():.6f}")
        
        # Check discount computation
        discount = discounts_model.discount(t_test, T_test, x_test, r0_test)
        print(f"\n  Discount D(t={t_test.item():.2f}, T={T_test.item():.2f}) = {discount.item():.6f}")
        # FIX: baseline should be exp(-r0 * T) since T is a tenor
        print(f"  Baseline exp(-r0*T) ≈ {torch.exp(-r0_test * T_test).item():.6f}")
        
        # Check if networks are initialized or stuck
        print("\n  ODE Network weights (first layer):")
        first_layer = list(discounts_model.u_ode.net.net[0].parameters())[0]
        print(f"    Mean: {first_layer.mean().item():.6f}, Std: {first_layer.std().item():.6f}")
        print(f"    Min: {first_layer.min().item():.6f}, Max: {first_layer.max().item():.6f}")
        
        print("\n  Volatility Network weights (first layer):")
        first_layer = list(x_P.rn.diff_net.net[0].parameters())[0]
        print(f"    Mean: {first_layer.mean().item():.6f}, Std: {first_layer.std().item():.6f}")
        
        # Test full curve
        print("\n  Testing full discount curve:")
        curve = discounts_model.discount_curve(t_test, tenors, x_test, r0_test)
        print(f"    Predicted discounts: {curve[:5].numpy()}")
        print(f"    Observed discounts:  {discount_data[0, :5].numpy()}")
    
    # ===========================
    # PLOTS: u(τ) and probs
    # ===========================
    # Build a two-row figure: (1) u(τ) dense + meeting markers, (2) jump probs at meeting τ
    rows = 2 if tau_grid.numel() > 0 else 1
    fig = make_subplots(
        rows=rows, cols=1, shared_xaxes=True, vertical_spacing=0.1,
        subplot_titles=(["u(τ) shape", "Jump probabilities at meeting times"] if rows == 2 else ["u(τ) shape"])
    )
    
    # Row 1: u(τ) dense
    fig.add_trace(
        go.Scatter(x=tau_dense.detach().cpu().numpy(),
                   y=u_dense.detach().cpu().numpy(),
                   mode='lines',
                   name='u(τ) (dense)'),
        row=1, col=1
    )
    
    # Meeting markers (if any)
    if tau_grid.numel() > 0:
        fig.add_trace(
            go.Scatter(x=tau_meet_np,
                       y=u_meet.detach().cpu().numpy(),
                       mode='markers',
                       name='u(τ) at meetings'),
            row=1, col=1
        )
        # vertical stems for meeting τ locations
        for tm in tau_meet_np:
            fig.add_shape(
                type="line",
                x0=tm, x1=tm,
                y0=min(float(u_dense.min()), float(u_meet.min())) if u_meet.numel()>0 else float(u_dense.min()),
                y1=max(float(u_dense.max()), float(u_meet.max())) if u_meet.numel()>0 else float(u_dense.max()),
                line=dict(width=1, dash="dot"),
                row=1, col=1
            )
    
    fig.update_xaxes(title_text="τ (years ahead from t)", row=rows, col=1)
    fig.update_yaxes(title_text="u(τ)", row=1, col=1)
    
    # Row 2: jump probabilities at meeting τ (lines over τ; one line per jump)
    if rows == 2 and probs_meet is not None and probs_meet.size > 0:
        jump_vals_bp = discounts_model.possible_jumps.detach().cpu().numpy() * 10_000
        for j in range(probs_meet.shape[1]):
            fig.add_trace(
                go.Scatter(x=tau_meet_np,
                           y=probs_meet[:, j],
                           mode='lines+markers',
                           name=f'P(J={jump_vals_bp[j]:.0f} bp)'),
                row=2, col=1
            )
        fig.update_yaxes(title_text="Probability", range=[0, 1], row=2, col=1)
    
    fig.update_layout(
        title=f"u(τ) and jump probabilities at t={float(t_test.item()):.2f}, T={float(T_test.item()):.2f}",
        template='plotly_white',
        hovermode='x unified',
        height=450 if rows == 1 else 700,
        showlegend=True
    )
    fig.show()
    
    print("="*60)

# Run diagnostics
diagnose_model(nd, phys_sde, ten, sr, ts, disc_mat)


MODEL DIAGNOSTICS

Test inputs:
  x = 0.000000
  t = 0.000000
  r0 = 0.052500
  T = 3.000000

  ODE u values at tau=[0.1 0.5 1. ]: [ 0.        -1.2001091 -1.9998605]

  Volatility σ(x=0.000) = 2.548214
  Drift f(x=0.000) = -5.477656

  Discount D(t=0.00, T=3.00) = 0.930073
  Baseline exp(-r0*T) ≈ 0.854277

  ODE Network weights (first layer):
    Mean: 0.140206, Std: 0.671572
    Min: -1.526616, Max: 1.266259

  Volatility Network weights (first layer):
    Mean: -0.064852, Std: nan

  Testing full discount curve:
    Predicted discounts: [0.9753423 0.9575349 0.9447047 0.9362545 0.9317031]
    Observed discounts:  [0.97531486 0.95318985 0.930529   0.907017   0.88478583]


/Users/josemelo/Desktop/dev/notebook/.conda/lib/python3.11/site-packages/torch/utils/_device.py:103: UserWarning:

std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)



In [187]:
# Check training history
print("\nTraining check:")
print(f"Did you see log-likelihood values during training?")
print(f"Did the loss decrease over epochs?")

# Quick sanity check - rerun one UKF step manually
print("\nManual UKF step test:")
x0 = torch.tensor([0.0])
P0 = torch.eye(1) * 0.01

with torch.no_grad():
    # Predict one discount curve
    t = ts[0]
    r0 = sr[0]
    pred = nd.discount_curve(t, ten, x0, r0)
    obs = disc_mat[0]
    
    print(f"Time t={t.item():.4f}")
    print(f"Short rate r0={r0.item():.4f}")
    print(f"\nFirst 5 tenors: {ten[:5].numpy()}")
    print(f"Predicted:      {pred[:5].numpy()}")
    print(f"Observed:       {obs[:5].numpy()}")
    print(f"Difference:     {(obs[:5] - pred[:5]).numpy()}")


Training check:
Did you see log-likelihood values during training?
Did the loss decrease over epochs?

Manual UKF step test:
Time t=0.0000
Short rate r0=0.0525

First 5 tenors: [0.5 1.  1.5 2.  2.5]
Predicted:      [0.9753423 0.9575349 0.9447047 0.9362545 0.9317031]
Observed:       [0.97531486 0.95318985 0.930529   0.907017   0.88478583]
Difference:     [-2.7418137e-05 -4.3450594e-03 -1.4175713e-02 -2.9237509e-02
 -4.6917260e-02]


In [188]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import torch

def analyze_trained_model(
    discounts_model: NeuralDiscounts,
    x_P: PhysicalSDE,
    short_rates: torch.Tensor,
    tenors: torch.Tensor,
    discount_data: torch.Tensor,
    times: torch.Tensor,
    dt: float = 1/360,
    plot_times_indices: list = None,
):
    N, M = discount_data.shape
    if plot_times_indices is None:
        plot_times_indices = [0, N//4, N//2, 3*N//4, N-1]

    print("Running forward pass through trained model...")

    x0 = torch.tensor([0.0], dtype=torch.float32, device=tenors.device)
    P0 = torch.eye(1, dtype=torch.float32, device=tenors.device) * 0.01
    R  = torch.eye(M, dtype=torch.float32, device=tenors.device) * 1e-3

    def fx(x):
        # deterministic step under P
        drift = x_P.f(torch.zeros_like(x), x) * dt
        return x + drift

    ukf = SquareRootUnscentedKalmanFilter(
        fx=fx, hx=lambda _x: _x,
        x=x0, P=P0, Q=torch.eye(1, dtype=torch.float32, device=tenors.device) * 1e-4, R=R,
        alpha=1, beta=2.0, kappa=0.0,
    )

    ukf_states = []
    predicted_discounts = []
    log_likelihoods = []

    with torch.no_grad():
        for i in range(N):
            r0 = short_rates[i]
            t_abs = times[i]

            # measurement model
            def hx_row(x, *, t=t_abs, r0_row=r0):
                d = discounts_model.discount_curve(t, tenors, x, r0_row)
                # safety: keep within (0,1] to avoid log issues later
                return d.clamp_(1e-8, 1.0)

            ukf.hx = lambda x, _hx=hx_row: _hx(x)

            # Q_t = sigma(x)^2 * dt
            sigma = x_P.g(t_abs.view(1,1), ukf.x.view(1,1)).squeeze()
            q_scalar = (sigma ** 2) * dt
            ukf.Q = q_scalar.view(1,1)

            z = discount_data[i, :]
            x, S, ll, z = ukf.step(z)

            ukf_states.append(x.clone())
            predicted_discounts.append(hx_row(x).clone())
            log_likelihoods.append(float(ll.item()))  # ensure plain float

            if i % 50 == 0:
                print(f"  Step {i}/{N}: x={float(x.item()):.4f}, ll={float(ll.item()):.2f}")

    print("Forward pass complete! Generating plots...")

    # ---- to numpy/CPU
    x_history          = torch.stack(ukf_states).squeeze(-1).cpu().numpy()
    times_np           = times.detach().cpu().numpy()
    tenors_np          = tenors.detach().cpu().numpy()
    pred_discounts_np  = torch.stack(predicted_discounts).detach().cpu().numpy()
    obs_discounts_np   = discount_data.detach().cpu().numpy()

    # ================================================================
    # Plot 1: State + Log-Likelihood
    # ================================================================
    fig1 = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Latent State x(t)', 'Log-Likelihood'),
        vertical_spacing=0.15
    )
    fig1.add_trace(go.Scatter(x=times_np, y=x_history, mode='lines', name='x(t)'), row=1, col=1)
    fig1.add_trace(go.Scatter(x=times_np, y=log_likelihoods, mode='lines', name='Log-Likelihood'), row=2, col=1)
    fig1.update_xaxes(title_text="Time (years)", row=2, col=1)
    fig1.update_yaxes(title_text="x(t)", row=1, col=1)
    fig1.update_yaxes(title_text="Log-Likelihood", row=2, col=1)
    fig1.update_layout(height=600, template='plotly_white', showlegend=False)
    fig1.show()

    # ================================================================
    # Plot 2: Yield Curves at Selected Times
    # ================================================================
    fig2 = go.Figure()
    colors = ['red', 'orange', 'green', 'blue', 'purple', 'brown', 'magenta']

    for idx_num, idx in enumerate(plot_times_indices):
        if idx < 0 or idx >= N:  # guard bad indices
            continue
        t_val = float(times[idx].item())
        obs_yields  = -np.log(obs_discounts_np[idx]) / tenors_np
        pred_yields = -np.log(pred_discounts_np[idx]) / tenors_np
        color = colors[idx_num % len(colors)]
        fig2.add_trace(go.Scatter(x=tenors_np, y=obs_yields,  mode='markers',
                                  name=f't={t_val:.2f}y (obs)',  marker=dict(size=8, color=color),
                                  legendgroup=f'group{idx}'))
        fig2.add_trace(go.Scatter(x=tenors_np, y=pred_yields, mode='lines',
                                  name=f't={t_val:.2f}y (model)', line=dict(width=3, dash='dash', color=color),
                                  legendgroup=f'group{idx}'))

    fig2.update_layout(title='Yield Curves: Observed vs Model',
                       xaxis_title='Maturity (years)', yaxis_title='Yield',
                       template='plotly_white', hovermode='x unified', height=500)
    fig2.show()

    # ================================================================
    # Plot 3: Jump Probabilities Evolution
    # ================================================================
    print("Computing jump probabilities...")
    maturity_for_probs = 2.0
    tenor_idx = (torch.abs(tenors - maturity_for_probs)).argmin()
    T_plot = float(tenors[tenor_idx].item())

    all_probs = []
    with torch.no_grad():
        for idx in range(len(ukf_states)):
            t = times[idx]
            x = ukf_states[idx]
            tau_grid = discounts_model.generate_meeting_times(t, tenors[tenor_idx])
            if tau_grid.numel() > 0:
                u = discounts_model.solve_u(t, tau_grid, x)
                if u.dim() == 1:
                    u = u.unsqueeze(-1)
                probs = discounts_model.jump_probs(u)          # (K, |J|)
                avg_probs = probs.mean(dim=0)
            else:
                avg_probs = torch.ones(len(discounts_model.possible_jumps), device=x.device) / len(discounts_model.possible_jumps)
            all_probs.append(avg_probs.detach().cpu().numpy())

    all_probs = np.asarray(all_probs)
    jump_values = discounts_model.possible_jumps.detach().cpu().numpy()

    fig3 = go.Figure()
    for j in range(all_probs.shape[1]):
        fig3.add_trace(go.Scatter(x=times_np, y=all_probs[:, j], mode='lines',
                                  name=f'Jump={jump_values[j]*10_000:.0f} bp'))
    fig3.update_layout(title=f'Jump Probabilities Over Time (T={T_plot:.2f}y)',
                       xaxis_title='Time (years)', yaxis_title='Probability',
                       template='plotly_white', hovermode='x unified',
                       height=500, yaxis=dict(range=[0,1]))
    fig3.show()

    # ================================================================
    # Plot 3A: u(τ) shape at selected times (same T=T_plot)
    # ================================================================
    num_rows = sum(1 for idx in plot_times_indices if 0 <= idx < N)
    if num_rows == 0:
        num_rows = 1
        valid_indices = []
    else:
        valid_indices = [idx for idx in plot_times_indices if 0 <= idx < N]

    fig3a = make_subplots(
        rows=num_rows, cols=1, shared_xaxes=False, vertical_spacing=0.07,
        subplot_titles=[f"t index {idx}" for idx in valid_indices] if valid_indices else ["u(τ)"]
    )

    row_counter = 0
    for idx in plot_times_indices:
        if idx < 0 or idx >= N:
            continue
        row_counter += 1

        t = times[idx]
        x = ukf_states[idx]
        T_this = tenors[tenor_idx]  # same maturity used in Plot 3
        T_scalar = float(T_this.item())

        # Dense u(τ) from 0..T_scalar
        discounts_model.u_ode.set_state(t=t, x=x)
        u0 = torch.zeros_like(x)
        tau_dense = torch.linspace(0.0, T_scalar, 200, dtype=x.dtype, device=x.device)
        u_dense = odeint(discounts_model.u_ode, u0, tau_dense, method='rk4',
                         options={'step_size': ODE_STEP}).squeeze()

        # u(τ) at meeting times
        tau_grid = discounts_model.generate_meeting_times(t, T_this)
        if tau_grid.numel() > 0:
            u_meet = discounts_model.solve_u(t, tau_grid, x)  # (K,)
            tau_meet_np = tau_grid.detach().cpu().numpy()
            u_meet_np = u_meet.detach().cpu().numpy()
            ymin = float(min(u_dense.min().item(), u_meet.min().item()))
            ymax = float(max(u_dense.max().item(), u_meet.max().item()))
        else:
            tau_meet_np = np.array([])
            u_meet_np = np.array([])
            ymin = float(u_dense.min().item())
            ymax = float(u_dense.max().item())

        # Plot dense curve
        fig3a.add_trace(
            go.Scatter(
                x=tau_dense.detach().cpu().numpy(),
                y=u_dense.detach().cpu().numpy(),
                mode='lines',
                name=f'u(τ) dense @ t#{idx}',
                showlegend=False
            ),
            row=row_counter, col=1
        )

        # Plot meeting markers and stems
        if tau_meet_np.size > 0:
            fig3a.add_trace(
                go.Scatter(
                    x=tau_meet_np,
                    y=u_meet_np,
                    mode='markers',
                    name='meetings',
                    showlegend=(row_counter == 1)
                ),
                row=row_counter, col=1
            )
            for tm in tau_meet_np:
                fig3a.add_shape(
                    type="line",
                    x0=tm, x1=tm, y0=ymin, y1=ymax,
                    line=dict(width=1, dash="dot"),
                    row=row_counter, col=1
                )

        fig3a.update_yaxes(title_text="u(τ)", row=row_counter, col=1)

    fig3a.update_xaxes(title_text="τ (years ahead from t)", row=row_counter if row_counter else 1, col=1)
    fig3a.update_layout(
        title=f"u(τ) shapes at selected times (T={T_plot:.2f}y)",
        template='plotly_white',
        hovermode='x unified',
        height=max(350, 230 * num_rows)
    )
    fig3a.show()

    # ================================================================
    # Plot 4: Fit Quality
    # ================================================================
    errors = obs_discounts_np - pred_discounts_np
    rmse_per_time  = np.sqrt((errors**2).mean(axis=1))
    rmse_per_tenor = np.sqrt((errors**2).mean(axis=0))

    fig4 = make_subplots(
        rows=2, cols=2,
        subplot_titles=('RMSE Over Time','RMSE by Maturity','Error Heatmap','Actual vs Predicted (All)'),
        specs=[[{"type":"scatter"},{"type":"scatter"}],[{"type":"heatmap"},{"type":"scatter"}]],
        vertical_spacing=0.12, horizontal_spacing=0.12
    )
    fig4.add_trace(go.Scatter(x=times_np, y=rmse_per_time, mode='lines', name='RMSE'), row=1, col=1)
    fig4.add_trace(go.Bar(x=tenors_np, y=rmse_per_tenor, name='RMSE'), row=1, col=2)
    step = max(1, N // 100)
    fig4.add_trace(go.Heatmap(z=errors[::step].T, x=times_np[::step], y=tenors_np,
                              colorscale='RdBu', zmid=0, colorbar=dict(x=0.46)), row=2, col=1)
    # scatter all points
    flat_obs  = obs_discounts_np.ravel()
    flat_pred = pred_discounts_np.ravel()
    fig4.add_trace(go.Scatter(x=flat_obs, y=flat_pred, mode='markers',
                              marker=dict(size=2, opacity=0.3), name='Data'), row=2, col=2)
    rng = [flat_obs.min(), flat_obs.max()]
    fig4.add_trace(go.Scatter(x=rng, y=rng, mode='lines', line=dict(dash='dash'), name='y=x'), row=2, col=2)

    fig4.update_xaxes(title_text="Time (years)", row=1, col=1)
    fig4.update_xaxes(title_text="Maturity (years)", row=1, col=2)
    fig4.update_xaxes(title_text="Time (years)", row=2, col=1)
    fig4.update_xaxes(title_text="Observed", row=2, col=2)
    fig4.update_yaxes(title_text="RMSE", row=1, col=1)
    fig4.update_yaxes(title_text="RMSE", row=1, col=2)
    fig4.update_yaxes(title_text="Maturity (years)", row=2, col=1)
    fig4.update_yaxes(title_text="Predicted", row=2, col=2)
    fig4.update_layout(height=800, template='plotly_white', showlegend=False,
                       title_text="Model Fit Quality")
    fig4.show()

    print("\n" + "="*60)
    print("MODEL PERFORMANCE SUMMARY")
    print("="*60)
    print(f"Overall RMSE: {np.sqrt((errors**2).mean()):.6f}")
    print(f"Mean Absolute Error: {np.abs(errors).mean():.6f}")
    print(f"Max Absolute Error: {np.abs(errors).max():.6f}")
    print(f"Total Log-Likelihood: {sum(log_likelihoods):.2f}")
    print(f"State x range: [{x_history.min():.4f}, {x_history.max():.4f}]")
    print("="*60)

    return {
        'states': ukf_states,
        'predicted_discounts': predicted_discounts,
        'log_likelihoods': log_likelihoods,
        'errors': errors
    }

# Example call (adjust indices to your N)
results = analyze_trained_model(
    discounts_model=nd,
    x_P=phys_sde,
    short_rates=sr,
    tenors=ten,
    discount_data=disc_mat,
    times=ts,
    dt=1/252,
    plot_times_indices=[0, 10, 20, 30, 40, 49]
)

Running forward pass through trained model...
  Step 0/1: x=0.1309, ll=23.65
Forward pass complete! Generating plots...


Computing jump probabilities...



MODEL PERFORMANCE SUMMARY
Overall RMSE: 0.098982
Mean Absolute Error: 0.087496
Max Absolute Error: 0.138925
Total Log-Likelihood: 23.65
State x range: [0.1309, 0.1309]


In [189]:
# %%
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go

DAYS_PER_YEAR = 360.0

def _years_to_days360(y: float) -> pd.Timedelta:
    return pd.to_timedelta(y * DAYS_PER_YEAR, unit="D")

def _abs_date_from_year_offset(base_date: pd.Timestamp, y: float) -> pd.Timestamp:
    return base_date + _years_to_days360(float(y))

def _next_k_dates_after(dates: pd.DatetimeIndex, t: pd.Timestamp, k: int):
    mask = dates > t
    idxs = np.where(mask)[0][:k]
    return dates[idxs]

def analyze_time_alignment_single_tau(
    *,
    discounts_df: pd.DataFrame,           # your 'discounts'
    meeting_dates: pd.DatetimeIndex,      # your 'meeting_dates'
    times: torch.Tensor,                  # your 'ts'
    discounts_model: NeuralDiscounts,     # your 'nd'
    t_idx: int = 0,                       # which observation index to inspect
    k_meetings: int = 12,                 # how many future meetings to compare
    T_min_years: float = 0.5,             # min horizon for the model
    pad_meetings_years: float = 0.01      # small padding on horizon
):
    """
    For a single date index t_idx:
      • Compute τ_real (years from t) for the next k real meetings
      • Compute τ_model from generate_meeting_times(t, T)
      • Plot both τ sets on a simple 2-lane axis (times-from-t only)
      • Return a comparison table with Δτ (years) and Δ days
    """
    # --- anchors & t selection ---
    base_discount_date = discounts_df.index[0]
    t_years = float(times[t_idx].item())
    t_date  = _abs_date_from_year_offset(base_discount_date, t_years)

    # --- real next meeting dates & their τ (years from t) ---
    real_next_dates = _next_k_dates_after(meeting_dates, t_date, k_meetings)
    tau_real = np.array([(d - t_date).days / DAYS_PER_YEAR for d in real_next_dates], dtype=float)

    # pick a model horizon that at least covers the last real τ
    if len(tau_real) > 0:
        T_horizon = max(T_min_years, float(tau_real[-1]) + pad_meetings_years)
    else:
        T_horizon = max(T_min_years, 2.0)

    # --- model τ grid ---
    with torch.no_grad():
        tau_model_torch = discounts_model.generate_meeting_times(
            torch.tensor([t_years], dtype=torch.float32),
            torch.tensor([T_horizon], dtype=torch.float32)
        ).detach().cpu()
    tau_model = tau_model_torch.numpy().astype(float)

    # --- comparison table (align by index) ---
    n_rows = max(len(tau_real), len(tau_model))
    real_list  = list(tau_real)  + [np.nan] * (n_rows - len(tau_real))
    model_list = list(tau_model) + [np.nan] * (n_rows - len(tau_model))

    # Δτ in years and days (days use 360-day year as in your construction)
    delta_tau_years = []
    delta_days = []
    for r, m in zip(real_list, model_list):
        if np.isnan(r) or np.isnan(m):
            delta_tau_years.append(np.nan)
            delta_days.append(np.nan)
        else:
            dty = float(m - r)
            delta_tau_years.append(dty)
            delta_days.append(int(round(dty * DAYS_PER_YEAR)))

    cmp_df = pd.DataFrame({
        "i": np.arange(1, n_rows + 1, dtype=int),
        "τ_real (years)": real_list,
        "τ_model (years)": model_list,
        "Δτ (years)": delta_tau_years,
        "Δ days (model - real)": delta_days
    })

    # --- simple times-from-t plot (no dates anywhere) ---
    fig = go.Figure()

    if len(tau_real) > 0:
        fig.add_trace(go.Scatter(
            x=tau_real, y=[0]*len(tau_real),
            mode="markers",
            marker=dict(size=10),
            name="Real τ (data)",
            hovertemplate="τ_real=%{x:.3f} y<extra></extra>"
        ))

    if len(tau_model) > 0:
        fig.add_trace(go.Scatter(
            x=tau_model, y=[1]*len(tau_model),
            mode="markers",
            marker=dict(size=10, symbol="diamond"),
            name="Model τ (implied)",
            hovertemplate="τ_model=%{x:.3f} y<extra></extra>"
        ))

    # layout
    fig.update_yaxes(
        tickmode="array",
        tickvals=[0, 1],
        ticktext=["Real τ", "Model τ"],
        range=[-0.5, 1.5],
        showgrid=False
    )
    fig.update_xaxes(title="τ (years ahead from t)")
    fig.update_layout(
        title=f"Meeting times-from-t, single index t_idx={t_idx} (T≈{T_horizon:.2f}y)",
        template="plotly_white",
        hovermode="x unified",
        height=300,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)
    )
    fig.show()

    # concise print
    print(f"t_idx={t_idx} | t (years from base) = {t_years:.6f}")
    if len(tau_real) >= 12 and len(tau_model) >= 12:
        print(f"  12th meeting Δτ: {delta_tau_years[11]:+.6f} years  ({delta_days[11]:+d} days)")
    elif len(tau_real) == 0 and len(tau_model) == 0:
        print("  No future meetings found within horizon.")
    else:
        print("  (<12 future meetings within horizon)")

    return {
        "t_idx": t_idx,
        "t_years": t_years,
        "T_horizon_years": T_horizon,
        "tau_real_years": tau_real,
        "tau_model_years": tau_model,
        "comparison": cmp_df
    }


res = analyze_time_alignment_single_tau(
    discounts_df=discounts,
    meeting_dates=meeting_dates,
    times=ts,
    discounts_model=nd,
    t_idx=0,       # pick the date you want
    k_meetings=12
)

# View the comparison table of τ's
res["comparison"]

t_idx=0 | t (years from base) = 0.000000
  (<12 future meetings within horizon)


,i,τ_real (years),τ_model (years),Δτ (years),Δ days (model - real)
0,1,0.075000,0.075000,2.980232e-09,0
1,2,0.191667,0.191667,-3.973643e-09,0
2,3,0.336111,0.336111,-1.258320e-08,0
3,4,0.444444,0.444444,3.311369e-09,0
4,5,0.580556,0.580556,2.649095e-09,0
5,6,0.697222,0.697222,1.059638e-08,0
